In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

<center> <img src="https://blog-assets.freshworks.com/freshdesk/wp-content/uploads/2020/06/18152022/Blog_Banner_v1-01-1024x410.jpg"> </center>

### About Dataset

#### Context

**Problem Statement**

Customer Personality Analysis is a detailed analysis of a company’s ideal customers. It helps a business to better understand its customers and makes it easier for them to modify products according to the specific needs, behaviors and concerns of different types of customers. 

Customer personality analysis helps a business to modify its product based on its target customers from different types of customer segments. For example, instead of spending money to market a new product to every customer in the company’s database, a company can analyze which customer segment is most likely to buy the product and then market the product only on that particular segment.


#### Content

**Attributes**

**People**

* ID: Customer's unique identifier
* Year_Birth: Customer's birth year
* Education: Customer's education level
* Marital_Status: Customer's marital status
* Income: Customer's yearly household income
* Kidhome: Number of children in customer's household
* Teenhome: Number of teenagers in customer's household
* Dt_Customer: Date of customer's enrollment with the company
* Recency: Number of days since customer's last purchase
* Complain: 1 if customer complained in the last 2 years, 0 otherwise

**Products**

* MntWines: Amount spent on wine in last 2 years
* MntFruits: Amount spent on fruits in last 2 years
* MntMeatProducts: Amount spent on meat in last 2 years
* MntFishProducts: Amount spent on fish in last 2 years
* MntSweetProducts: Amount spent on sweets in last 2 years
* MntGoldProds: Amount spent on gold in last 2 years

**Promotion**

* NumDealsPurchases: Number of purchases made with a discount
* AcceptedCmp1: 1 if customer accepted the offer in the 1st campaign, 0 otherwise
* AcceptedCmp2: 1 if customer accepted the offer in the 2nd campaign, 0 otherwise
* AcceptedCmp3: 1 if customer accepted the offer in the 3rd campaign, 0 otherwise
* AcceptedCmp4: 1 if customer accepted the offer in the 4th campaign, 0 otherwise
* AcceptedCmp5: 1 if customer accepted the offer in the 5th campaign, 0 otherwise
* Response: 1 if customer accepted the offer in the last campaign, 0 otherwise

**Place**

* NumWebPurchases: Number of purchases made through the company’s web site
* NumCatalogPurchases: Number of purchases made using a catalogue
* NumStorePurchases: Number of purchases made directly in stores
* NumWebVisitsMonth: Number of visits to company’s web site in the last month

#### Target

Need to perform clustering to summarize customer segments.

In [ ]:
# Importing all the necessary libraries

import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Reading and making a copy of the dataset

main_df = pd.read_csv("/kaggle/input/customer-personality-analysis/marketing_campaign.csv", sep="\t")
df = main_df.copy()
df.head()

### EDA

In [ ]:
# Checking the shape of the dataset

df.shape

In [ ]:
# Finding the basic information regarding dataset

df.info()

* Here we have only 3 object type datatype and rest are numerical.

In [ ]:
# Finding the number of unique values present in each column

df.nunique()

**NOTE** 
* In above cell "Z_CostContact" and "Z_Revenue" have same value in all the rows that's why they are not going to contribute anything in the model building. So we can drop them.  

In [ ]:
# Checking if any NaN is present in column or not

df.isna().any()

* Income column have some missing value in it so we will need to fill it by by either mean or median.

In [ ]:
# Checking for null value using heatmap

sns.heatmap(df.isnull())

In [ ]:
# Dropping columns because they will not contribute anything in model building

df=df.drop(columns=["Z_CostContact", "Z_Revenue"],axis=1)
df.head()

In [ ]:
# Finding the correlation between the feature column

plt.figure(figsize=(20,20))
sns.heatmap(df.corr(), annot=True)
plt.show()

* No two columns are too much correlated with each other so we can't drop any column on the basis of correlation.

In [ ]:
# Checking for correlation by unstacking data

corr = df.corr()
c1 = corr.abs().unstack()
c1.sort_values(ascending = False)[24:50:2]

* It is used to calculate how one variable is correlated/ dependent on other variable.
* Extreme values signify high correlation.
* Multicollinear variables with correlation more than a threshold are usually dropped from the dataset.

### Preprocessing of the dataset

In [ ]:
# Filling the missing value in the income my mean

df['Income'] = df['Income'].fillna(df['Income'].mean())
df.isna().any() 

In [ ]:
df.head()

In [ ]:
# Checking number of unique categories present in the "Marital_Status"

df['Marital_Status'].value_counts()  

In [ ]:
df['Marital_Status'] = df['Marital_Status'].replace(['Married', 'Together'],'relationship')
df['Marital_Status'] = df['Marital_Status'].replace(['Divorced', 'Widow', 'Alone', 'YOLO', 'Absurd'],'Single')

* In the above cell we are grouping 'Married', 'Together' as "relationship" 
* Whereas 'Divorced', 'Widow', 'Alone', 'YOLO', 'Absurd' as "Single"

In [ ]:
# Count of different values present in Marital_Status

df['Marital_Status'].value_counts()  

In [ ]:
# Combining different dataframe into a single column to reduce the number of dimension

df['Kids'] = df['Kidhome'] + df['Teenhome']
df['Expenses'] = df['MntWines'] + df['MntFruits'] + df['MntMeatProducts'] + df['MntFishProducts'] + df['MntSweetProducts'] + df['MntGoldProds']
df['TotalAcceptedCmp'] = df['AcceptedCmp1'] + df['AcceptedCmp2'] + df['AcceptedCmp3'] + df['AcceptedCmp4'] + df['AcceptedCmp5'] + df['Response']
df['NumTotalPurchases'] = df['NumWebPurchases'] + df['NumCatalogPurchases'] + df['NumStorePurchases'] + df['NumDealsPurchases']

In [ ]:
# Deleting some column to reduce dimension and complexity of model

col_del = ["AcceptedCmp1" , "AcceptedCmp2", "AcceptedCmp3" , "AcceptedCmp4","AcceptedCmp5", "Response","NumWebVisitsMonth", "NumWebPurchases","NumCatalogPurchases","NumStorePurchases","NumDealsPurchases" , "Kidhome", "Teenhome","MntWines", "MntFruits", "MntMeatProducts", "MntFishProducts", "MntSweetProducts", "MntGoldProds"]
df=df.drop(columns=col_del,axis=1)
df.head()

In [ ]:
# Adding a column "Age" in the dataframe

df['Age'] = 2015 - df["Year_Birth"]

In [ ]:
df['Education'].value_counts()

In [ ]:
# Changing category into UG and PG only

df['Education'] = df['Education'].replace(['PhD','2n Cycle','Graduation', 'Master'],'PG')  
df['Education'] = df['Education'].replace(['Basic'], 'UG')

In [ ]:
# Number of days a customer was engaged with company

# Changing Dt_customer into timestamp format
df['Dt_Customer'] = pd.to_datetime(df.Dt_Customer)
df['first_day'] = '01-01-2015'
df['first_day'] = pd.to_datetime(df.first_day)
df['day_engaged'] = (df['first_day'] - df['Dt_Customer']).dt.days

In [ ]:
df=df.drop(columns=["ID", "Dt_Customer", "first_day", "Year_Birth", "Dt_Customer", "Recency", "Complain"],axis=1)
df.shape

### Visualization

In [ ]:
fig = px.bar(df, x='Marital_Status', y='Expenses', color="Education")
fig.show()

In [ ]:
fig = px.bar(df, x='Marital_Status', y='Expenses', color="Marital_Status")
fig.show()

**Less number of single customer**

In [ ]:
fig = px.histogram (df, x = "Expenses",  facet_row = "Marital_Status",  template = 'plotly_dark')
fig.show ()

In [ ]:
fig = px.histogram (df, x = "Expenses",  facet_row = "Education",  template = 'plotly_dark')
fig.show ()

In [ ]:
fig = px.histogram (df, x = "NumTotalPurchases",  facet_row = "Education",  template = 'plotly_dark')
fig.show ()

In [ ]:
fig = px.histogram (df, x = "Age",  facet_row = "Marital_Status",  template = 'plotly_dark')
fig.show ()

In [ ]:
fig = px.histogram (df, x = "Income",  facet_row = "Marital_Status",  template = 'plotly_dark')
fig.show ()

In [ ]:
fig =  px.pie (df, names = "Marital_Status", hole = 0.4, template = "gridon")
fig.show ()

**35% of the customer are single whereas more 64% are in relationship.**

In [ ]:
fig =  px.pie (df, names = "Education", hole = 0.4, template = "plotly_dark")
fig.show ()

** More than 97% customer are from PG background. and Approx. 2% are from UG.

In [ ]:
sns.barplot(x = df['Expenses'],y = df['Education']);
plt.title('Total Expense based on the Education Level');

In [ ]:
sns.barplot(x = df['Income'],y = df['Education']);
plt.title('Total Income based on the Education Level');

In [ ]:
df.describe()

In [ ]:
sns.heatmap(df.corr(), annot=True)

In [ ]:
cate = []
for i in df.columns:
    if (df[i].dtypes == "object"):
        cate.append(i)

print(cate)

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [ ]:
df['Marital_Status'].value_counts()

### Label Encoding

In [ ]:
lbl_encode = LabelEncoder()
for i in cate:
    df[i]=df[[i]].apply(lbl_encode.fit_transform)

In [ ]:
df1 = df.copy()

### Standardization 

In [ ]:
scaled_features = StandardScaler().fit_transform(df1.values)
scaled_features_df = pd.DataFrame(scaled_features, index=df1.index, columns=df1.columns)

In [ ]:
scaled_features_df.head()

### Elbow Method 

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
wcss=[]
for i in range (1,11):
 kmeans=KMeans(n_clusters=i,init='k-means++',random_state=42)
 kmeans.fit(scaled_features_df)
 wcss.append(kmeans.inertia_)
plt.figure(figsize=(16,8))
plt.plot(range(1,11),wcss, 'bx-')
plt.title('The Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS')
plt.show()

**As it is not very clear from the elbow method that which value of K to choose.**

### Silhouette Score

In [ ]:
from sklearn.metrics import silhouette_score 

In [ ]:

silhouette_scores = []
for i in range(2,10):
    m1=KMeans(n_clusters=i, random_state=42)
    c = m1.fit_predict(scaled_features_df)
    silhouette_scores.append(silhouette_score(scaled_features_df, m1.fit_predict(scaled_features_df))) 
plt.bar(range(2,10), silhouette_scores) 
plt.xlabel('Number of clusters', fontsize = 20) 
plt.ylabel('S(i)', fontsize = 20) 
plt.show()

**Here we are using Silhouette score to measure the value of K**

In [ ]:
silhouette_scores

In [ ]:
# Getting the maximum value of silhouette score and adding 2 in index because index starts from 2.

sc=max(silhouette_scores)
number_of_clusters=silhouette_scores.index(sc)+2
print("Number of Cluster Required is : ", number_of_clusters)

### Model Building

In [ ]:
# Training a predicting using K-Means Algorithm.

kmeans=KMeans(n_clusters=number_of_clusters, random_state=42).fit(scaled_features_df)
pred=kmeans.predict(scaled_features_df)


# Appending those cluster value into main dataframe (without standard-scalar)

df['cluster'] = pred + 1

In [ ]:
df.head()

In [ ]:
scaled_features_df.head()

In [ ]:
df['Education'].value_counts()

* 0 means PG and 1 means UG
* There are very less customer from UG background

### Clustering 

In [ ]:
pl = sns.countplot(x=df["cluster"])
pl.set_title("Distribution Of The Clusters")
plt.show()

**Note :-**

**As we can see here that weightage of customer are more in cluster 1 as compare to other.**

In [ ]:
# Clusters interpretation 
sns.set(rc={'axes.facecolor':'black', 'figure.facecolor':'black', 'axes.grid' : False, 'font.family': 'Ubuntu'})

for i in df:
    diag = sns.FacetGrid(df, col = "cluster", hue = "cluster", palette = "Set1")
    diag.map(plt.hist, i, bins=6, ec="k") 
    diag.set_xticklabels(rotation=25, color = 'white')
    diag.set_yticklabels(color = 'white')
    diag.set_xlabels(size=16, color = 'white')
    diag.set_titles(size=16, color = '#f01132', fontweight="bold")
    diag.fig.set_figheight(6)

### Report 

#### Based on above information we can divide customer into 3 parts:- 

1. **Highly Active Customer** :- These customers belong to cluster one.
2. **Moderately Active Customer** :- These customers belong to cluster two.
3. **Least Active Customer** :-  These customers belong to cluster third.

#### Characteristics of Highly Active Customer

- **In terms of Education**
 - Highly Active Customer are from PG background


- **In terms of Marital_status**
 - Number of people in relationship are approx. two times of single people


- **In terms of Income**
 - Income of Highly active customer are little less as compare to Moderately active customer.
 
 
- **In terms of Kids**
 - Highly active customer have more number of children as compare to other customer ( avg. of 1 child ).
 
 
- **In terms of Expenses**
 - Expenses of Highly Active customer are less as compare to moderate.
 - These customer spent avg. of approx. 100-200 unit money.


- **In terms of Age**
 - Age of these customer are between 25 to 75.
 - Maximum customer age are between 40 to 50.


- **In terms of day_engaged**
 - Highly Active customer are more loyal as they engaged with company for longer period of time.

#### Characteristics of Moderately Active Customer

- **In terms of Education**
 - Moderately Active Customer are also from PG backgroud


- **In terms of Marital_status**
 - Number of people in relationship are slightly more as compare to single people


- **In terms of Income**
 - Income of Moderately active customer are higher as compare to other customer.


- **In terms of Kids**
 - Moderately active customer have less number of childern as compare to highly active customer ( Max. customer has no child ).


- **In terms of Expenses**
 - Expenses of Moderately Active customer are more as compare to Active.
 - These customer spent avg. of approx. 500-2000 unit money.


- **In terms of Age**
 - Age of these customer are between 25 to 75.
 - Maximum customer age are between 35 to 60.


- **In terms of day_engaged**
 - Moderately Active customer are slightly less engaged with company as compare to Highly Active Customer.
 

#### Characteristics of Least Active Customer

- **In terms of Education**
 - Least Active Customer are from UG backgroud

    
- **In terms of Marital_status**
 - Number of people in relationship are approx. equal to single people

- **In terms of Income**
 - Income of Least active customer are very less or say negligible.
    
- **In terms of Kids**
 - Only few of these customer have child.

- **In terms of Expenses**
 - Expenses of Least Active customer are very less or say negligible.


- **In terms of Age**
 - Age of these customer are between 15 to 30.


- **In terms of day_engaged**
 - Least Active customer are not much enrolled with company for longer time.

**Please like this notebook👍, If it helped you in learning something new🙂 and do check my other notebook.**